# 🎯 Real / Fake / Deepfake Images Detection

## Stratégie globale
- **Phase 1** : Real vs AI-generated (Dataset 1) → entraîne 3 modèles : BaseCNN, ResNet50, XceptionNet
- **Phase 2** : Real vs Deepfake (Dataset 2) → fine-tune les 3 modèles unifiés
- **Comparaison finale** → choix du modèle de production

# 🟦 PHASE 1 : Real vs AI-Generated
## Étape 1 : Data Exploration (Dataset 1)

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
from collections import defaultdict

In [ ]:
DATASET_PATH = "/kaggle/input/datasets/tristanzhang32/ai-generated-images-vs-real-images"
# ====================== 2. CHEMINS EXACTS ======================
train_real_path = os.path.join(DATASET_PATH, "train", "real")
train_fake_path = os.path.join(DATASET_PATH, "train", "fake")
test_real_path  = os.path.join(DATASET_PATH, "test",  "real")
test_fake_path  = os.path.join(DATASET_PATH, "test",  "fake")

In [ ]:
def count_images(folder):
    if not os.path.exists(folder):
        return 0
    jpg = len(glob.glob(os.path.join(folder, "*.jpg"))) + len(glob.glob(os.path.join(folder, "*.jpeg")))
    png = len(glob.glob(os.path.join(folder, "*.png")))
    return jpg + png

# Comptages
train_real = count_images(train_real_path)
train_fake = count_images(train_fake_path)
test_real  = count_images(test_real_path)
test_fake  = count_images(test_fake_path)

total_real = train_real + test_real
total_fake = train_fake + test_fake
total_images = total_real + total_fake

In [ ]:
# ====================== 4. TABLEAU RÉCAPITULATIF (pour la validation) ======================
summary_data = {
    'Split': ['Train', 'Test', 'TOTAL'],
    'Real': [train_real, test_real, total_real],
    'Fake/AI': [train_fake, test_fake, total_fake],
    '% Real': [f"{train_real/(train_real+train_fake)*100:.1f}%" if train_real+train_fake>0 else "0%", 
               f"{test_real/(test_real+test_fake)*100:.1f}%" if test_real+test_fake>0 else "0%", 
               f"{total_real/total_images*100:.1f}%"],
    '% Fake/AI': [f"{train_fake/(train_real+train_fake)*100:.1f}%" if train_real+train_fake>0 else "0%", 
                  f"{test_fake/(test_real+test_fake)*100:.1f}%" if test_real+test_fake>0 else "0%", 
                  f"{total_fake/total_images*100:.1f}%"],
    'Total images': [train_real + train_fake, test_real + test_fake, total_images]
}

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))

In [ ]:
# ====================== 5. GRAPH BALANCE (Train + Test + Global) ======================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Train
axes[0].bar(['Real', 'Fake/AI'], [train_real, train_fake], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Train Split', fontsize=14, fontweight='bold')
axes[0].set_ylabel("Nombre d'images")

# Test
axes[1].bar(['Real', 'Fake/AI'], [test_real, test_fake], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Test Split', fontsize=14, fontweight='bold')

# Global
axes[2].bar(['Real', 'Fake/AI'], [total_real, total_fake], color=['#2ecc71', '#e74c3c'])
axes[2].set_title('Global Balance', fontsize=14, fontweight='bold')

for ax in axes:
    ax.grid(axis='y', alpha=0.3)
    for i, v in enumerate(ax.patches):
        ax.text(v.get_x() + v.get_width()/2, v.get_height() + 500, f'{int(v.get_height()):,}', 
                ha='center', va='bottom', fontweight='bold')

plt.suptitle('Balance du Dataset : Real vs AI-Generated (Fake)', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ====================== 6. SAMPLES VISUELS (4 par catégorie) ======================
def plot_samples(title, real_path, fake_path, color_real='#2ecc71', color_fake='#e74c3c'):
    real_samples = glob.glob(os.path.join(real_path, "*.[jJ][pP][gG]"))[:4] + \
                   glob.glob(os.path.join(real_path, "*.[pP][nN][gG]"))[:4]
    fake_samples = glob.glob(os.path.join(fake_path, "*.[jJ][pP][gG]"))[:4] + \
                   glob.glob(os.path.join(fake_path, "*.[pP][nN][gG]"))[:4]
    
    samples = real_samples[:4] + fake_samples[:4]
    labels = ['REAL']*4 + ['FAKE/AI']*4
    colors = [color_real]*4 + [color_fake]*4
    
    plt.figure(figsize=(16, 8))
    for i, (img_path, label, col) in enumerate(zip(samples, labels, colors)):
        img = Image.open(img_path)
        plt.subplot(2, 4, i+1)
        plt.imshow(img)
        plt.title(label, color=col, fontsize=14, fontweight='bold')
        plt.axis('off')
    plt.suptitle(title, fontsize=18, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("\n Exemples visuels :")
plot_samples("TRAIN SPLIT - Exemples Real vs Fake/AI", train_real_path, train_fake_path)
plot_samples("TEST SPLIT - Exemples Real vs Fake/AI", test_real_path, test_fake_path)

print("   • Dataset équilibré (50/50)")
print("   • Structure claire (train/test)")

## Étape 2 : Data Preparation

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import glob
import os
from tqdm import tqdm
from PIL import ImageFile

In [ ]:
#pour fixer les images tronqués dans les datasets de kaggle
ImageFile.LOAD_TRUNCATED_IMAGES = True
print(" FIX images tronquées activé")

In [ ]:
# ====================== 1. TRANSFORMS  ======================
# Standardisation => toutes les images 224x224
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),           
    # Standard pour ResNet / ViT
    transforms.RandomHorizontalFlip(p=0.5),  
    # Data Augmentation => pour augmenter la variété (généraliser pas mémoriser)
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# ====================== 2. CUSTOM DATASET  ======================
class AIFakeDataset(Dataset):
    def __init__(self, root_dir, transform=None, split='train'):
        self.transform = transform
        self.samples = []
        
        real_dir = os.path.join(root_dir, split, 'real')
        fake_dir = os.path.join(root_dir, split, 'fake')
        
        # Real = label 0
        for img_path in glob.glob(os.path.join(real_dir, "*.[jJ][pP][gG]")) + \
                        glob.glob(os.path.join(real_dir, "*.[pP][nN][gG]")):
            self.samples.append((img_path, 0))
        
        # Fake/AI = label 1
        for img_path in glob.glob(os.path.join(fake_dir, "*.[jJ][pP][gG]")) + \
                        glob.glob(os.path.join(fake_dir, "*.[pP][nN][gG]")):
            self.samples.append((img_path, 1))
        
        print(f" {split.capitalize()} dataset chargé : {len(self.samples):,} images "
              f"({len([s for s in self.samples if s[1]==0])} Real / {len([s for s in self.samples if s[1]==1])} Fake)")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

In [ ]:
# ====================== 3. CRÉATION DES DATASETS & DATALOADERS ======================
train_dataset = AIFakeDataset(DATASET_PATH, transform=train_transform, split='train')
test_dataset  = AIFakeDataset(DATASET_PATH, transform=test_transform,  split='test')

batch_size = 64
#le modèle reçoit 64 images d’un coup -> 1 batch = 64 images + leurs labels
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                          num_workers=4, #4 processus chargent les données en parallèle (plus rapide)
                          pin_memory=True,
                          persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, 
                         num_workers=4, pin_memory=True)

print(f"\n DataLoaders prêts !")
print(f"   • Train batches : {len(train_loader)}")
print(f"   • Test batches  : {len(test_loader)}")
print(f"   • Images par batch : {batch_size}")

# ====================== 4. TEST (1 batch) ======================
# pour vérifier que les images arrivent bien jusqu’au modèle, dans le bon format
images, labels = next(iter(train_loader))
print(f"\n Test batch shape : {images.shape} | Labels : {labels.shape}")
print(f"   • Exemple label : {'Fake/AI' if labels[0].item() == 1 else 'Real'}")

### 🆕 Amélioration : Split Train → Train + Validation

Le dataset original n'a que `train` et `test`. Pour pouvoir détecter l'overfitting **pendant** l'entraînement, on crée un **set de validation** depuis le train (15%, stratifié).

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset
import numpy as np

# On split les indices du train_dataset
all_labels = [s[1] for s in train_dataset.samples]
train_idx, val_idx = train_test_split(
    np.arange(len(train_dataset)),
    test_size=0.15,
    stratify=all_labels,
    random_state=42
)

train_subset = Subset(train_dataset, train_idx)
val_subset   = Subset(train_dataset, val_idx)

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_subset,   batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"✅ Splits créés :")
print(f"   • Train : {len(train_subset):,} images")
print(f"   • Val   : {len(val_subset):,} images")
print(f"   • Test  : {len(test_dataset):,} images")

---
## 🆕 Fonctions génériques améliorées (réutilisables pour TOUS les modèles)

Ces fonctions vont être utilisées pour BaseCNN, ResNet50 et XceptionNet.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ====================== TRAIN UNE EPOCH ======================
def train_one_epoch(model, loader, criterion, optimizer, epoch_num, total_epochs, tag=""):
    """Entraîne 1 epoch et retourne (loss, accuracy)."""
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch_num+1}/{total_epochs} [{tag}] train")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        pbar.set_postfix(loss=running_loss/total, acc=100.*correct/total)
    
    return running_loss / total, 100. * correct / total

# ====================== ÉVAL SIMPLE PENDANT TRAIN (loss + acc) ======================
@torch.no_grad()
def quick_eval(model, loader, criterion):
    """Évaluation rapide pendant le training (loss + acc seulement)."""
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return running_loss / total, 100. * correct / total

In [ ]:
# ====================== TRAINING LOOP COMPLET (avec validation) ======================
def train_with_validation(model, train_loader, val_loader, criterion, optimizer,
                          scheduler, num_epochs, model_name="model", save_path=None):
    """
    Entraîne le modèle avec validation à chaque epoch.
    Retourne l'historique complet pour tracer les courbes.
    Sauvegarde le best model (basé sur val_acc).
    """
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss':   [], 'val_acc':   []
    }
    best_val_acc = 0.0
    
    print(f"\n{'='*60}")
    print(f"🚀 Training: {model_name}")
    print(f"{'='*60}")
    
    for epoch in range(num_epochs):
        # Train
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion,
                                           optimizer, epoch, num_epochs, model_name)
        # Validation
        val_loss, val_acc = quick_eval(model, val_loader, criterion)
        
        if scheduler is not None:
            scheduler.step()
        
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f"  Epoch {epoch+1}/{num_epochs} | "
              f"train_loss={tr_loss:.4f} train_acc={tr_acc:.2f}% | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.2f}%")
        
        # Sauvegarde best model
        if val_acc > best_val_acc and save_path is not None:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f"    ✅ Best model saved (val_acc={val_acc:.2f}%)")
    
    return history

In [ ]:
# ====================== ÉVALUATION COMPLÈTE (toutes les métriques) ======================
@torch.no_grad()
def complete_evaluation(model, loader, model_name="Model", class_names=['Real', 'Fake']):
    """
    Évalue le modèle avec TOUTES les métriques recommandées pour CNN classification binaire.
    Retourne un dict avec toutes les métriques + génère les graphiques.
    """
    model.eval()
    all_preds, all_probs, all_targets = [], [], []
    
    for images, labels in tqdm(loader, desc=f"Evaluation {model_name}"):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]   # proba classe 1 (Fake)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_targets.extend(labels.cpu().numpy())
    
    all_preds   = np.array(all_preds)
    all_probs   = np.array(all_probs)
    all_targets = np.array(all_targets)
    
    # ===== Calcul des métriques =====
    metrics = {
        'accuracy':  accuracy_score(all_targets, all_preds),
        'precision': precision_score(all_targets, all_preds),
        'recall':    recall_score(all_targets, all_preds),
        'f1':        f1_score(all_targets, all_preds),
        'auc':       roc_auc_score(all_targets, all_probs),
    }
    
    # ===== Affichage =====
    print(f"\n{'='*50}")
    print(f"📊 ÉVALUATION COMPLÈTE — {model_name}")
    print(f"{'='*50}")
    print(f"  Accuracy  : {metrics['accuracy']:.4f}")
    print(f"  Precision : {metrics['precision']:.4f}")
    print(f"  Recall    : {metrics['recall']:.4f}")
    print(f"  F1-score  : {metrics['f1']:.4f}")
    print(f"  AUC-ROC   : {metrics['auc']:.4f}")
    print(f"\n📋 Classification Report :")
    print(classification_report(all_targets, all_preds, target_names=class_names))
    
    # ===== Graphiques =====
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Confusion Matrix
    cm = confusion_matrix(all_targets, all_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=axes[0],
                cbar=False, annot_kws={'size': 14})
    axes[0].set_title(f'{model_name} — Confusion Matrix', fontweight='bold')
    axes[0].set_xlabel('Prédit'); axes[0].set_ylabel('Vrai')
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(all_targets, all_probs)
    axes[1].plot(fpr, tpr, linewidth=2, label=f"AUC = {metrics['auc']:.4f}")
    axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'{model_name} — ROC Curve', fontweight='bold')
    axes[1].legend(loc='lower right'); axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    metrics['preds']   = all_preds
    metrics['probs']   = all_probs
    metrics['targets'] = all_targets
    return metrics

In [ ]:
# ====================== COURBES D'APPRENTISSAGE + DÉTECTION OVERFIT/UNDERFIT ======================
def plot_learning_curves(history, model_name="Model"):
    """
    Trace les courbes train vs val pour Loss et Accuracy.
    Analyse automatiquement overfitting / underfitting.
    """
    epochs = range(1, len(history['train_loss']) + 1)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss
    axes[0].plot(epochs, history['train_loss'], 'o-', label='Train Loss', color='#3498db', linewidth=2)
    axes[0].plot(epochs, history['val_loss'],   's-', label='Val Loss',   color='#e74c3c', linewidth=2)
    axes[0].set_title(f'{model_name} — Loss', fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(alpha=0.3)
    
    # Accuracy
    axes[1].plot(epochs, history['train_acc'], 'o-', label='Train Acc', color='#3498db', linewidth=2)
    axes[1].plot(epochs, history['val_acc'],   's-', label='Val Acc',   color='#e74c3c', linewidth=2)
    axes[1].set_title(f'{model_name} — Accuracy', fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # ===== ANALYSE AUTO OVERFIT / UNDERFIT =====
    final_train_acc = history['train_acc'][-1]
    final_val_acc   = history['val_acc'][-1]
    gap = final_train_acc - final_val_acc
    
    print(f"\n🔍 Analyse {model_name} :")
    print(f"   • Train accuracy finale : {final_train_acc:.2f}%")
    print(f"   • Val accuracy finale   : {final_val_acc:.2f}%")
    print(f"   • Gap (train - val)     : {gap:+.2f}%")
    
    if gap > 10:
        print(f"   ⚠️  OVERFITTING détecté (gap > 10%)")
        print(f"      → Recommandations : plus de dropout, plus d'augmentation, early stopping, plus de données")
    elif final_train_acc < 75:
        print(f"   ⚠️  UNDERFITTING possible (train_acc < 75%)")
        print(f"      → Recommandations : modèle plus complexe, plus d'epochs, lr plus élevé")
    else:
        print(f"   ✅ Bon équilibre train/val (pas d'overfitting majeur)")

In [ ]:
# ====================== TEST D'UNE IMAGE AVEC AFFICHAGE VRAIE CLASSE ======================
def test_single_image(model, dataset, idx, model_name="Model",
                      class_names=['REAL', 'FAKE/AI'], normalize_mean=[0.485, 0.456, 0.406],
                      normalize_std=[0.229, 0.224, 0.225]):
    """
    Teste une image et affiche : image, vraie classe, classe prédite, risk score, confidence.
    Marque clairement si la prédiction est correcte ou non.
    """
    model.eval()
    image_tensor, true_label = dataset[idx]
    
    with torch.no_grad():
        image_input = image_tensor.unsqueeze(0).to(device)
        output = model(image_input)
        probs = torch.softmax(output, dim=1)[0]
        pred_label = probs.argmax().item()
        prob_fake = probs[1].item()
        confidence = probs.max().item() * 100
        risk_score = int(prob_fake * 100)
    
    # Dénormalisation pour affichage
    img_to_show = image_tensor.permute(1, 2, 0).cpu().numpy()
    img_to_show = (img_to_show * np.array(normalize_std) + np.array(normalize_mean)).clip(0, 1)
    
    is_correct = (pred_label == true_label)
    status = "✅ CORRECT" if is_correct else "❌ WRONG"
    color  = 'green' if is_correct else 'red'
    
    plt.figure(figsize=(8, 8))
    plt.imshow(img_to_show)
    plt.title(f"{model_name} — {status}\n"
              f"Vraie classe : {class_names[true_label]}  |  "
              f"Prédite : {class_names[pred_label]}\n"
              f"Fake Risk Score : {risk_score}/100  |  Confidence : {confidence:.1f}%",
              fontsize=12, fontweight='bold', color=color)
    plt.axis('off')
    plt.show()
    
    return {
        'true': class_names[true_label],
        'pred': class_names[pred_label],
        'correct': is_correct,
        'risk_score': risk_score,
        'confidence': confidence
    }

---
## Étape 3 : Modèle 1 — BaseCNN (from scratch)

🆕 **Améliorations** : training avec validation, courbes d'apprentissage, métriques complètes, Grad-CAM.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Device : {device}")

In [ ]:
class BaseCNN(nn.Module):
    def __init__(self):
        super(BaseCNN, self).__init__()
        # Couches convolutives simples (from scratch)
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),   # 224x224 -> 224x224
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # -> 112x112
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # -> 56x56
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # -> 28x28
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # -> 14x14
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 2)   # 2 classes : Real (0) / Fake/AI (1)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model_scratch = BaseCNN().to(device)

In [ ]:
# ====================== ENTRAÎNEMENT BASECNN AVEC VALIDATION ======================
import torch.optim as optim

model_basecnn = BaseCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_basecnn.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

NUM_EPOCHS_BASECNN = 6

history_basecnn = train_with_validation(
    model_basecnn, train_loader, val_loader,
    criterion, optimizer, scheduler,
    num_epochs=NUM_EPOCHS_BASECNN,
    model_name="BaseCNN",
    save_path="/kaggle/working/basecnn_phase1_best.pth"
)

In [ ]:
# Courbes + analyse overfit/underfit
plot_learning_curves(history_basecnn, "BaseCNN")

In [ ]:
# Évaluation complète sur le test set
# On charge le best model sauvegardé pendant le training
model_basecnn.load_state_dict(torch.load("/kaggle/working/basecnn_phase1_best.pth"))
metrics_basecnn = complete_evaluation(model_basecnn, test_loader, "BaseCNN — Phase 1")

In [ ]:
# Test sur quelques images avec affichage vraie classe vs prédite
import random
random.seed(42)
test_indices = random.sample(range(len(test_dataset)), 4)

print("🧪 Test BaseCNN sur 4 images aléatoires :\n")
for idx in test_indices:
    test_single_image(model_basecnn, test_dataset, idx,
                      model_name="BaseCNN", class_names=['REAL', 'FAKE/AI'])

---
## Étape 4 : Modèle 2 — ResNet50 avec **vrai fine-tuning progressif**

🆕 **Stratégie correcte de fine-tuning** :
- **Phase A (2 epochs)** : on **gèle** le backbone ImageNet, on entraîne uniquement la tête classifier (lr=1e-3)
- **Phase B (4 epochs)** : on **dégèle** tout, on fine-tune l'ensemble (lr=1e-4)

Cela évite que les gradients d'une tête aléatoire détruisent les features pré-entraînés.

In [ ]:
from torchvision import models

# ====================== 1. CHARGEMENT RESNET50 + REMPLACEMENT TÊTE ======================
model_resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
num_features = model_resnet.fc.in_features
model_resnet.fc = nn.Linear(num_features, 2)   # Real (0) / Fake (1)
model_resnet = model_resnet.to(device)

# Compter les paramètres
total = sum(p.numel() for p in model_resnet.parameters())
print(f"ResNet50 — {total/1e6:.2f}M paramètres total")

In [ ]:
# ====================== PHASE A : BACKBONE GELÉ, TRAIN HEAD ONLY ======================
print("\n🔒 PHASE A — Backbone gelé, on entraîne uniquement la tête classifier")

# Geler tout le backbone
for param in model_resnet.parameters():
    param.requires_grad = False

# Dégeler uniquement la couche fc (classifier)
for param in model_resnet.fc.parameters():
    param.requires_grad = True

# Optimizer ne contient QUE les paramètres entraînables
optimizer_phaseA = optim.Adam(filter(lambda p: p.requires_grad, model_resnet.parameters()),
                               lr=1e-3, weight_decay=1e-4)

trainable = sum(p.numel() for p in model_resnet.parameters() if p.requires_grad)
print(f"Paramètres entraînables (Phase A) : {trainable/1e6:.2f}M / {total/1e6:.2f}M")

history_resnet_A = train_with_validation(
    model_resnet, train_loader, val_loader,
    criterion=nn.CrossEntropyLoss(),
    optimizer=optimizer_phaseA,
    scheduler=None,
    num_epochs=2,
    model_name="ResNet50 [Phase A]",
    save_path="/kaggle/working/resnet50_phaseA_best.pth"
)

In [ ]:
# ====================== PHASE B : FULL FINE-TUNING ======================
print("\n🔓 PHASE B — Dégel complet, full fine-tuning avec lr faible")

# Dégeler TOUT le réseau
for param in model_resnet.parameters():
    param.requires_grad = True

optimizer_phaseB = optim.Adam(model_resnet.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_phaseB = optim.lr_scheduler.CosineAnnealingLR(optimizer_phaseB, T_max=4)

history_resnet_B = train_with_validation(
    model_resnet, train_loader, val_loader,
    criterion=nn.CrossEntropyLoss(),
    optimizer=optimizer_phaseB,
    scheduler=scheduler_phaseB,
    num_epochs=4,
    model_name="ResNet50 [Phase B]",
    save_path="/kaggle/working/resnet50_phase1_best.pth"
)

# Concaténer l'historique pour avoir une vue complète
history_resnet = {
    k: history_resnet_A[k] + history_resnet_B[k]
    for k in history_resnet_A
}
print(f"\n📈 Historique total : {len(history_resnet['train_loss'])} epochs (2 Phase A + 4 Phase B)")

In [ ]:
# Courbes complètes (Phase A + Phase B)
plot_learning_curves(history_resnet, "ResNet50 (Fine-tuning progressif)")

# Annotation visuelle de la transition Phase A → Phase B
import matplotlib.pyplot as plt
epochs = range(1, len(history_resnet['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, key, ylabel in [(axes[0], 'loss', 'Loss'), (axes[1], 'acc', 'Accuracy (%)')]:
    ax.plot(epochs, history_resnet[f'train_{key}'], 'o-', label=f'Train {ylabel}', color='#3498db')
    ax.plot(epochs, history_resnet[f'val_{key}'],   's-', label=f'Val {ylabel}',   color='#e74c3c')
    ax.axvline(2.5, color='gray', linestyle='--', alpha=0.7, label='Phase A → B')
    ax.set_title(f'ResNet50 — {ylabel} (avec transition Phase)', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Évaluation complète ResNet50
model_resnet.load_state_dict(torch.load("/kaggle/working/resnet50_phase1_best.pth"))
metrics_resnet = complete_evaluation(model_resnet, test_loader, "ResNet50 — Phase 1")

In [ ]:
# Test ResNet50 sur les mêmes 4 images
print("🧪 Test ResNet50 sur les mêmes 4 images :\n")
for idx in test_indices:
    test_single_image(model_resnet, test_dataset, idx,
                      model_name="ResNet50", class_names=['REAL', 'FAKE/AI'])

---
## Étape 5 : Modèle 3 — **XceptionNet** (référence en deepfake detection)

🆕 **XceptionNet** est l'architecture de référence pour la détection de deepfake (paper FaceForensics++). Ses **depthwise separable convolutions** sont particulièrement efficaces pour capter les artefacts de génération.

On utilise `timm` qui fournit Xception pré-entraîné sur ImageNet.

In [ ]:
# Installation de timm si pas déjà fait
!pip install -q timm

In [ ]:
import timm

# ====================== 1. CHARGEMENT XCEPTION ======================
model_xception = timm.create_model('xception', pretrained=True, num_classes=2)
model_xception = model_xception.to(device)

total = sum(p.numel() for p in model_xception.parameters())
print(f"XceptionNet — {total/1e6:.2f}M paramètres total")

In [ ]:
# ====================== PHASE A : Head only ======================
print("\n🔒 PHASE A — Backbone gelé, train head only")

for param in model_xception.parameters():
    param.requires_grad = False
# La tête classifier de Xception (timm) est accessible via .get_classifier()
for param in model_xception.get_classifier().parameters():
    param.requires_grad = True

optimizer_phaseA = optim.Adam(filter(lambda p: p.requires_grad, model_xception.parameters()),
                               lr=1e-3, weight_decay=1e-4)

trainable = sum(p.numel() for p in model_xception.parameters() if p.requires_grad)
print(f"Paramètres entraînables (Phase A) : {trainable/1e6:.2f}M / {total/1e6:.2f}M")

history_xception_A = train_with_validation(
    model_xception, train_loader, val_loader,
    criterion=nn.CrossEntropyLoss(),
    optimizer=optimizer_phaseA,
    scheduler=None,
    num_epochs=2,
    model_name="Xception [Phase A]",
    save_path="/kaggle/working/xception_phaseA_best.pth"
)

In [ ]:
# ====================== PHASE B : Full fine-tuning ======================
print("\n🔓 PHASE B — Dégel complet")

for param in model_xception.parameters():
    param.requires_grad = True

optimizer_phaseB = optim.Adam(model_xception.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_phaseB = optim.lr_scheduler.CosineAnnealingLR(optimizer_phaseB, T_max=4)

history_xception_B = train_with_validation(
    model_xception, train_loader, val_loader,
    criterion=nn.CrossEntropyLoss(),
    optimizer=optimizer_phaseB,
    scheduler=scheduler_phaseB,
    num_epochs=4,
    model_name="Xception [Phase B]",
    save_path="/kaggle/working/xception_phase1_best.pth"
)

history_xception = {k: history_xception_A[k] + history_xception_B[k] for k in history_xception_A}

In [ ]:
# Courbes Xception
plot_learning_curves(history_xception, "XceptionNet (Fine-tuning progressif)")

In [ ]:
# Évaluation complète Xception
model_xception.load_state_dict(torch.load("/kaggle/working/xception_phase1_best.pth"))
metrics_xception = complete_evaluation(model_xception, test_loader, "XceptionNet — Phase 1")

In [ ]:
# Test Xception sur les mêmes 4 images
print("🧪 Test XceptionNet sur les mêmes 4 images :\n")
for idx in test_indices:
    test_single_image(model_xception, test_dataset, idx,
                      model_name="XceptionNet", class_names=['REAL', 'FAKE/AI'])

---
## Étape 6 : 🆕 **Grad-CAM** sur plusieurs images pour TOUS les modèles

Grad-CAM révèle les zones de l'image qui activent le plus la décision. On l'applique sur **plusieurs images** (pas juste une) et sur **les 3 modèles** pour comparer leur focus.

In [ ]:
import cv2
import numpy as np

# ====================== CLASSE GRAD-CAM GÉNÉRIQUE ======================
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.fwd_hook = target_layer.register_forward_hook(self.save_activation)
        self.bwd_hook = target_layer.register_full_backward_hook(self.save_gradient)
    
    def save_activation(self, module, input, output):
        self.activations = output.detach()
    
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    
    def remove_hooks(self):
        self.fwd_hook.remove()
        self.bwd_hook.remove()
    
    def generate_heatmap(self, input_tensor, class_idx):
        self.model.zero_grad()
        output = self.model(input_tensor)
        output[0, class_idx].backward(retain_graph=True)
        
        gradients = self.gradients[0]
        activations = self.activations[0]
        weights = gradients.mean(dim=[1, 2], keepdim=True)
        heatmap = (weights * activations).sum(dim=0).cpu().numpy()
        heatmap = np.maximum(heatmap, 0)
        heatmap = heatmap / (heatmap.max() + 1e-8)
        heatmap = heatmap.squeeze()
        heatmap = cv2.resize(heatmap, (224, 224))
        return heatmap

In [ ]:
# ====================== HELPER POUR REPÉRER LA TARGET LAYER ======================
def get_target_layer(model, model_name):
    """Retourne la dernière couche convolutive selon l'architecture."""
    if model_name.lower().startswith('basecnn'):
        return model.features[-3]   # Avant-dernière conv (avant le pool final)
    elif model_name.lower().startswith('resnet'):
        return model.layer4[-1]     # Dernier block ResNet
    elif model_name.lower().startswith('xception'):
        return model.conv4          # Dernière conv Xception (timm)
    else:
        raise ValueError(f"Modèle non supporté pour Grad-CAM : {model_name}")

In [ ]:
# ====================== AFFICHAGE GRAD-CAM POUR PLUSIEURS IMAGES ======================
def show_gradcam_multi(model, model_name, dataset, indices,
                       class_names=['REAL', 'FAKE/AI'],
                       normalize_mean=[0.485, 0.456, 0.406],
                       normalize_std=[0.229, 0.224, 0.225]):
    """
    Applique Grad-CAM sur plusieurs images et affiche image + heatmap côte à côte.
    Indique aussi vraie classe vs prédite.
    """
    model.eval()
    target_layer = get_target_layer(model, model_name)
    grad_cam = GradCAM(model, target_layer)
    
    n = len(indices)
    fig, axes = plt.subplots(2, n, figsize=(5*n, 10))
    if n == 1:
        axes = axes.reshape(2, 1)
    
    for col, idx in enumerate(indices):
        img_tensor, true_label = dataset[idx]
        input_tensor = img_tensor.unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = model(input_tensor)
            probs = torch.softmax(output, dim=1)[0]
            pred_label = probs.argmax().item()
            risk = int(probs[1].item() * 100)
            confidence = probs.max().item() * 100
        
        # Grad-CAM sur la classe prédite
        heatmap = grad_cam.generate_heatmap(input_tensor, pred_label)
        
        # Image originale
        img_to_show = img_tensor.permute(1, 2, 0).cpu().numpy()
        img_to_show = (img_to_show * np.array(normalize_std) + np.array(normalize_mean)).clip(0, 1)
        
        is_correct = pred_label == true_label
        status = "✅" if is_correct else "❌"
        color = 'green' if is_correct else 'red'
        
        axes[0, col].imshow(img_to_show)
        axes[0, col].set_title(f"{status} True: {class_names[true_label]} | Pred: {class_names[pred_label]}\n"
                               f"Risk: {risk}/100 | Conf: {confidence:.1f}%",
                               color=color, fontweight='bold')
        axes[0, col].axis('off')
        
        # Image + Grad-CAM overlay
        axes[1, col].imshow(img_to_show)
        axes[1, col].imshow(heatmap, cmap='jet', alpha=0.5)
        axes[1, col].set_title(f"Grad-CAM heatmap")
        axes[1, col].axis('off')
    
    plt.suptitle(f"Grad-CAM — {model_name}", fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    grad_cam.remove_hooks()

In [ ]:
# ====================== APPLIQUER GRAD-CAM SUR LES 3 MODÈLES ======================
# On utilise les MÊMES indices d'images pour comparer les 3 modèles

# ResNet50
show_gradcam_multi(model_resnet, "ResNet50", test_dataset, test_indices)

In [ ]:
# XceptionNet
show_gradcam_multi(model_xception, "XceptionNet", test_dataset, test_indices)

In [ ]:
# BaseCNN (pour comparaison — souvent moins informatif sur un CNN simple)
show_gradcam_multi(model_basecnn, "BaseCNN", test_dataset, test_indices)

**💡 Interprétation Grad-CAM** :
- **Sur fake AI** : les heatmaps doivent activer les zones d'artefacts (textures incohérentes, arrière-plans flous, mains anormales)
- **Sur real** : activation diffuse, équilibrée
- ResNet50 et XceptionNet (pré-entraînés) auront des heatmaps plus propres que BaseCNN (from scratch)

---
## Étape 7 : 🆕 **Tableau comparatif Phase 1** — Choix du meilleur modèle

In [ ]:
import pandas as pd

comparison_phase1 = pd.DataFrame({
    'Model': ['BaseCNN (scratch)', 'ResNet50 (fine-tuned)', 'XceptionNet (fine-tuned)'],
    'Accuracy':  [metrics_basecnn['accuracy'],  metrics_resnet['accuracy'],  metrics_xception['accuracy']],
    'Precision': [metrics_basecnn['precision'], metrics_resnet['precision'], metrics_xception['precision']],
    'Recall':    [metrics_basecnn['recall'],    metrics_resnet['recall'],    metrics_xception['recall']],
    'F1-score':  [metrics_basecnn['f1'],        metrics_resnet['f1'],        metrics_xception['f1']],
    'AUC':       [metrics_basecnn['auc'],       metrics_resnet['auc'],       metrics_xception['auc']],
}).round(4)

print("=" * 70)
print("📊 COMPARAISON PHASE 1 — Real vs AI-Generated")
print("=" * 70)
print(comparison_phase1.to_string(index=False))

# Bar plot comparatif
fig, ax = plt.subplots(figsize=(12, 6))
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-score', 'AUC']
x = np.arange(len(comparison_phase1))
width = 0.15
colors = ['#3498db', '#2ecc71', '#9b59b6', '#e67e22', '#e74c3c']

for i, metric in enumerate(metrics_to_plot):
    ax.bar(x + i*width, comparison_phase1[metric], width, label=metric, color=colors[i])

ax.set_xticks(x + 2*width)
ax.set_xticklabels(comparison_phase1['Model'], rotation=15)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Comparaison des modèles — Phase 1', fontsize=14, fontweight='bold')
ax.legend(loc='lower right'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

best_idx = comparison_phase1['F1-score'].idxmax()
best_model_name = comparison_phase1.loc[best_idx, 'Model']
print(f"\n🏆 Meilleur modèle Phase 1 (basé sur F1) : {best_model_name}")

---
# 🟪 PHASE 2 : Real vs Deepfake Faces (Dataset 2)

🆕 **Stratégie unifiée** : on **fine-tune les 3 modèles entraînés en Phase 1** sur le dataset Deepfake → modèles capables de détecter les **deux types de fake**.

## Étape 1 : Data Exploration Dataset 2

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
DATASET_PATH = "/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces"

In [ ]:
# Chargement des CSV
train_df = pd.read_csv(os.path.join(DATASET_PATH, "train.csv"))
valid_df = pd.read_csv(os.path.join(DATASET_PATH, "valid.csv"))
test_df  = pd.read_csv(os.path.join(DATASET_PATH, "test.csv"))

print(f"Train  : {len(train_df):,} images")
print(f"Valid  : {len(valid_df):,} images")
print(f"Test   : {len(test_df):,} images")

In [ ]:
# Répartition des classes
print("\nRépartition des classes :")
print(train_df['label'].value_counts().rename({0: "Real", 1: "Fake/Deepfake"}))

In [ ]:
def show_samples(df, title, n=4):
    plt.figure(figsize=(16, 6))
    df = df.reset_index(drop=True)
    for i in range(n):
        # Ajout du sous-dossier manquant
        img_path = os.path.join(DATASET_PATH, "real_vs_fake/real-vs-fake", df.iloc[i]['path'])
        if not os.path.exists(img_path):
            print(f" Image non trouvée : {img_path}")
            continue
        img = Image.open(img_path)
        plt.subplot(2, n, i+1)
        plt.imshow(img)
        label = "REAL" if df.iloc[i]['label'] == 0 else "FAKE/DEEPFAKE"
        plt.title(label, color='green' if df.iloc[i]['label']==0 else 'red')
        plt.axis('off')
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.show()

show_samples(train_df[train_df['label']==0].head(4), "Exemples REAL faces")
show_samples(train_df[train_df['label']==1].head(4), "Exemples FAKE/DEEPFAKE faces")

## Étape 2 : Data Preparation Dataset 2

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import os

In [ ]:
# Fix images tronquées
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [ ]:
DATASET_PATH = "/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces"

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class DeepfakeFacesDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.df = pd.read_csv(os.path.join(root_dir, csv_file))
        self.root_dir = root_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Ajout du sous-dossier manquant
        img_path = os.path.join(self.root_dir, "real_vs_fake/real-vs-fake", row['path'])
        image = Image.open(img_path).convert('RGB')
        label = int(row['label'])   # 0 = Real, 1 = Fake/Deepfake
        
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

# Création des datasets
train_dataset = DeepfakeFacesDataset("train.csv", DATASET_PATH, transform=train_transform)
valid_dataset = DeepfakeFacesDataset("valid.csv", DATASET_PATH, transform=test_transform)
test_dataset  = DeepfakeFacesDataset("test.csv",  DATASET_PATH, transform=test_transform)

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

print(f" Datasets créés : Train {len(train_dataset):,} | Valid {len(valid_dataset):,} | Test {len(test_dataset):,}")

In [ ]:
# Renommer les loaders Phase 2 pour éviter d'écraser ceux de Phase 1
train_loader_p2 = train_loader
valid_loader_p2 = valid_loader
test_loader_p2  = test_loader

# Et garder le test dataset Phase 2 sous un autre nom
test_dataset_p2 = test_dataset

print("✅ Loaders Phase 2 prêts :")
print(f"   • Train : {len(train_loader_p2.dataset):,} images")
print(f"   • Valid : {len(valid_loader_p2.dataset):,} images")
print(f"   • Test  : {len(test_loader_p2.dataset):,} images")

---
## Étape 3 : 🆕 **Fine-tuning des 3 modèles** (Phase 1 → Phase 2)

On charge les modèles entraînés en Phase 1 et on les fine-tune sur le dataset Deepfake.

**Important** : `lr` faible (1e-4) car on ne veut pas effacer les features apprises en Phase 1.

### 3.1 Fine-tuning BaseCNN unifié

In [ ]:
# Charger le BaseCNN de Phase 1
model_basecnn_unified = BaseCNN().to(device)
model_basecnn_unified.load_state_dict(torch.load("/kaggle/working/basecnn_phase1_best.pth"))
print("✅ BaseCNN Phase 1 chargé")

optimizer = optim.Adam(model_basecnn_unified.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

history_basecnn_p2 = train_with_validation(
    model_basecnn_unified, train_loader_p2, valid_loader_p2,
    criterion=nn.CrossEntropyLoss(),
    optimizer=optimizer, scheduler=scheduler,
    num_epochs=5,
    model_name="BaseCNN unifié",
    save_path="/kaggle/working/basecnn_unified_best.pth"
)

In [ ]:
plot_learning_curves(history_basecnn_p2, "BaseCNN unifié (Phase 2 fine-tune)")

In [ ]:
# Évaluation finale BaseCNN unifié
model_basecnn_unified.load_state_dict(torch.load("/kaggle/working/basecnn_unified_best.pth"))
metrics_basecnn_p2 = complete_evaluation(
    model_basecnn_unified, test_loader_p2,
    "BaseCNN unifié — Phase 2",
    class_names=['Real', 'Deepfake']
)

### 3.2 Fine-tuning ResNet50 unifié

In [ ]:
# Charger ResNet50 de Phase 1
model_resnet_unified = models.resnet50(weights=None)
model_resnet_unified.fc = nn.Linear(model_resnet_unified.fc.in_features, 2)
model_resnet_unified.load_state_dict(torch.load("/kaggle/working/resnet50_phase1_best.pth"))
model_resnet_unified = model_resnet_unified.to(device)
print("✅ ResNet50 Phase 1 chargé")

# Fine-tune avec lr faible (le réseau est déjà bien entraîné)
optimizer = optim.Adam(model_resnet_unified.parameters(), lr=5e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

history_resnet_p2 = train_with_validation(
    model_resnet_unified, train_loader_p2, valid_loader_p2,
    criterion=nn.CrossEntropyLoss(),
    optimizer=optimizer, scheduler=scheduler,
    num_epochs=5,
    model_name="ResNet50 unifié",
    save_path="/kaggle/working/resnet50_unified_best.pth"
)

In [ ]:
plot_learning_curves(history_resnet_p2, "ResNet50 unifié (Phase 2 fine-tune)")

In [ ]:
# Évaluation ResNet50 unifié
model_resnet_unified.load_state_dict(torch.load("/kaggle/working/resnet50_unified_best.pth"))
metrics_resnet_p2 = complete_evaluation(
    model_resnet_unified, test_loader_p2,
    "ResNet50 unifié — Phase 2",
    class_names=['Real', 'Deepfake']
)

### 3.3 Fine-tuning XceptionNet unifié

In [ ]:
# Charger Xception de Phase 1
model_xception_unified = timm.create_model('xception', pretrained=False, num_classes=2)
model_xception_unified.load_state_dict(torch.load("/kaggle/working/xception_phase1_best.pth"))
model_xception_unified = model_xception_unified.to(device)
print("✅ XceptionNet Phase 1 chargé")

optimizer = optim.Adam(model_xception_unified.parameters(), lr=5e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

history_xception_p2 = train_with_validation(
    model_xception_unified, train_loader_p2, valid_loader_p2,
    criterion=nn.CrossEntropyLoss(),
    optimizer=optimizer, scheduler=scheduler,
    num_epochs=5,
    model_name="Xception unifié",
    save_path="/kaggle/working/xception_unified_best.pth"
)

In [ ]:
plot_learning_curves(history_xception_p2, "XceptionNet unifié (Phase 2 fine-tune)")

In [ ]:
# Évaluation Xception unifié
model_xception_unified.load_state_dict(torch.load("/kaggle/working/xception_unified_best.pth"))
metrics_xception_p2 = complete_evaluation(
    model_xception_unified, test_loader_p2,
    "XceptionNet unifié — Phase 2",
    class_names=['Real', 'Deepfake']
)

---
## Étape 4 : 🆕 Tests + Grad-CAM sur Deepfake Faces

In [ ]:
# Sélection d'images aléatoires Phase 2
random.seed(42)
test_indices_p2 = random.sample(range(len(test_dataset_p2)), 4)

print("🧪 Test des 3 modèles unifiés sur les mêmes 4 images Deepfake :\n")
print("─" * 60)
print("BaseCNN unifié :\n")
for idx in test_indices_p2:
    test_single_image(model_basecnn_unified, test_dataset_p2, idx,
                      model_name="BaseCNN unifié", class_names=['REAL', 'DEEPFAKE'])

In [ ]:
print("─" * 60)
print("ResNet50 unifié :\n")
for idx in test_indices_p2:
    test_single_image(model_resnet_unified, test_dataset_p2, idx,
                      model_name="ResNet50 unifié", class_names=['REAL', 'DEEPFAKE'])

In [ ]:
print("─" * 60)
print("XceptionNet unifié :\n")
for idx in test_indices_p2:
    test_single_image(model_xception_unified, test_dataset_p2, idx,
                      model_name="XceptionNet unifié", class_names=['REAL', 'DEEPFAKE'])

In [ ]:
# Grad-CAM Phase 2 — particulièrement informatif sur les visages
show_gradcam_multi(model_resnet_unified, "ResNet50 unifié", test_dataset_p2,
                   test_indices_p2, class_names=['REAL', 'DEEPFAKE'])

In [ ]:
show_gradcam_multi(model_xception_unified, "XceptionNet unifié", test_dataset_p2,
                   test_indices_p2, class_names=['REAL', 'DEEPFAKE'])

**💡 Interprétation Grad-CAM sur visages** :
- Real : activation diffuse, équilibrée sur l'ensemble du visage
- Deepfake : zones de **blending** (frontière du visage), **yeux** (asymétries), **bouche** (artefacts dentaires) typiquement activées
- XceptionNet est généralement plus précis sur ces zones que ResNet50

---
# 🏆 Étape Finale : Tableau comparatif global & Choix du modèle de production

In [ ]:
# ====================== TABLEAU COMPARATIF GLOBAL (Phase 1 + Phase 2) ======================
final_comparison = pd.DataFrame({
    'Model': [
        'BaseCNN (P1)',  'BaseCNN unifié (P2)',
        'ResNet50 (P1)', 'ResNet50 unifié (P2)',
        'Xception (P1)', 'Xception unifié (P2)',
    ],
    'Phase': ['Phase 1', 'Phase 2', 'Phase 1', 'Phase 2', 'Phase 1', 'Phase 2'],
    'Accuracy': [
        metrics_basecnn['accuracy'],   metrics_basecnn_p2['accuracy'],
        metrics_resnet['accuracy'],    metrics_resnet_p2['accuracy'],
        metrics_xception['accuracy'],  metrics_xception_p2['accuracy'],
    ],
    'Precision': [
        metrics_basecnn['precision'],   metrics_basecnn_p2['precision'],
        metrics_resnet['precision'],    metrics_resnet_p2['precision'],
        metrics_xception['precision'],  metrics_xception_p2['precision'],
    ],
    'Recall': [
        metrics_basecnn['recall'],   metrics_basecnn_p2['recall'],
        metrics_resnet['recall'],    metrics_resnet_p2['recall'],
        metrics_xception['recall'],  metrics_xception_p2['recall'],
    ],
    'F1-score': [
        metrics_basecnn['f1'],   metrics_basecnn_p2['f1'],
        metrics_resnet['f1'],    metrics_resnet_p2['f1'],
        metrics_xception['f1'],  metrics_xception_p2['f1'],
    ],
    'AUC': [
        metrics_basecnn['auc'],   metrics_basecnn_p2['auc'],
        metrics_resnet['auc'],    metrics_resnet_p2['auc'],
        metrics_xception['auc'],  metrics_xception_p2['auc'],
    ],
}).round(4)

print("=" * 80)
print("📊 TABLEAU COMPARATIF GLOBAL")
print("=" * 80)
print(final_comparison.to_string(index=False))
final_comparison.to_csv("/kaggle/working/final_comparison.csv", index=False)
print("\n✅ Sauvegardé dans /kaggle/working/final_comparison.csv")

In [ ]:
# ====================== HEATMAP GLOBALE DES PERFORMANCES ======================
fig, ax = plt.subplots(figsize=(10, 6))
metrics_cols = ['Accuracy', 'Precision', 'Recall', 'F1-score', 'AUC']
heatmap_data = final_comparison.set_index('Model')[metrics_cols]
sns.heatmap(heatmap_data, annot=True, fmt='.4f', cmap='RdYlGn',
            vmin=0.5, vmax=1.0, ax=ax, cbar_kws={'label': 'Score'})
ax.set_title('Performances globales — Phase 1 & Phase 2', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ====================== BAR CHART COMPARATIF (F1) ======================
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#3498db' if 'P1' in m else '#e74c3c' for m in final_comparison['Model']]
bars = ax.bar(final_comparison['Model'], final_comparison['F1-score'], color=colors)
ax.set_title('F1-score par modèle (bleu = Phase 1, rouge = Phase 2 unifié)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('F1-score'); ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=20)
for bar, val in zip(bars, final_comparison['F1-score']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.3f}',
            ha='center', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ====================== RECOMMANDATION FINALE ======================
# On recommande le meilleur modèle UNIFIÉ (Phase 2) car il gère les deux types de fake

p2_models = final_comparison[final_comparison['Phase'] == 'Phase 2'].copy()
best_p2_idx = p2_models['F1-score'].idxmax()
best_unified = p2_models.loc[best_p2_idx]

print("=" * 70)
print("🎯 RECOMMANDATION POUR LE DÉPLOIEMENT")
print("=" * 70)
print(f"\n🏆 Meilleur modèle unifié (capable de détecter les 2 types de fake) :")
print(f"   → {best_unified['Model']}")
print(f"\n   Métriques :")
print(f"     • Accuracy  : {best_unified['Accuracy']:.4f}")
print(f"     • Precision : {best_unified['Precision']:.4f}")
print(f"     • Recall    : {best_unified['Recall']:.4f}")
print(f"     • F1-score  : {best_unified['F1-score']:.4f}")
print(f"     • AUC       : {best_unified['AUC']:.4f}")
print(f"\n💡 Justification : ce modèle a été entraîné sur le dataset 1 (AI-generated),")
print(f"   puis fine-tuné sur le dataset 2 (deepfake). Il capture donc les deux familles d'artefacts.")
print(f"\n📁 Fichier checkpoint : /kaggle/working/{best_unified['Model'].split()[0].lower()}_unified_best.pth")

---
## 📋 Conclusion

| Élément | Status |
|---|---|
| 3 modèles entraînés (BaseCNN, ResNet50, XceptionNet) | ✅ |
| Fine-tuning progressif correct (Phase A → B) | ✅ |
| Validation à chaque epoch (détection overfitting) | ✅ |
| Métriques complètes (Accuracy, Precision, Recall, F1, AUC) | ✅ |
| Confusion Matrix + ROC curve | ✅ |
| Courbes d'apprentissage train vs val | ✅ |
| Test image avec vraie classe vs prédite | ✅ |
| Grad-CAM sur plusieurs images & 3 modèles | ✅ |
| Modèles unifiés Phase 1 → Phase 2 | ✅ |
| Tableau comparatif global | ✅ |

### Prochaines étapes possibles
- Étendre à la **vidéo** : appliquer le modèle frame-by-frame ou utiliser CNN+LSTM
- **Ensembling** : moyenner les probabilités des 3 modèles unifiés (+ 1-2% AUC typiquement)
- **Test cross-dataset** : vérifier la généralisation
- **Compression** : ONNX / TensorRT pour le déploiement temps-réel